# 金融數據分析期中報告：效率前緣計算流程

**學號：611336023**

本 notebook 依老師公告要求，固定共同三家標的「台達電（2308）、富邦金（2881）、長榮（2603）」，並加入個人第四家標的「藥華藥（6446）」，建構：

1. 三家公司完整 mean-variance frontier  
2. 四家公司完整 mean-variance frontier  
3. 三家公司與四家公司疊合圖  

注意：此處的「效率前緣」依本次作業口徑畫的是**完整 mean-variance frontier**，包含最小變異組合上下兩支曲線，而不是只畫嚴格有效前緣的上半部。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pathlib import Path
import math

# 若在本機或 Colab 執行，請將本 notebook 與四個股價資料檔放在同一資料夾。
DATA_DIR = Path(".")

STUDENT_ID = "611336023"

ASSET_FILES = {
    "2308 台達電": DATA_DIR / "2024年_2308_台達電股價.xlsx",
    "2881 富邦金": DATA_DIR / "2024年_2881_富邦金股價.xlsx",
    "2603 長榮": DATA_DIR / "2024年_2603_長榮股價.xlsx",
    "6446 藥華藥": DATA_DIR / "2024年_6446_藥華藥股價.xlsx",
}

COMMON_3 = ["2308 台達電", "2881 富邦金", "2603 長榮"]
ASSETS_4 = ["2308 台達電", "2881 富邦金", "2603 長榮", "6446 藥華藥"]


## 一、讀取股價資料並依日期對齊

每一檔股票資料包含證券代碼、年月日、收盤價。  
本流程先讀取四檔股票的收盤價，再用共同交易日期進行合併，確保三家與四家分析使用相同資料期間與相同頻率。


In [ ]:
def read_price_file(path, asset_name):
    df = pd.read_excel(path)
    df["年月日"] = pd.to_datetime(df["年月日"].astype(str), format="%Y%m%d")
    df["收盤價(元)"] = pd.to_numeric(df["收盤價(元)"], errors="coerce")
    return df[["年月日", "收盤價(元)"]].rename(columns={"收盤價(元)": asset_name})

price_frames = []
for asset, path in ASSET_FILES.items():
    price_frames.append(read_price_file(path, asset))

prices = price_frames[0]
for pf in price_frames[1:]:
    prices = prices.merge(pf, on="年月日", how="inner")

prices = prices.sort_values("年月日").reset_index(drop=True)

print("價格資料期間：", prices["年月日"].min().date(), "到", prices["年月日"].max().date())
print("共同交易日數：", len(prices))
prices.head()


## 二、計算簡單日報酬率

本課程效率前緣推導採用**簡單報酬率**，因為投資組合報酬可以用各資產報酬率依權重線性加總：

`R_t = P_t / P_{t-1} - 1`


In [ ]:
returns = prices.set_index("年月日")[ASSETS_4].pct_change().dropna()

# 年化因子使用本資料實際日報酬筆數；本資料為 241 筆。
ANNUAL_FACTOR = len(returns)

print("日報酬資料期間：", returns.index.min().date(), "到", returns.index.max().date())
print("日報酬筆數：", ANNUAL_FACTOR)

returns.head()


## 三、計算年化平均報酬與年化變異數—共變異數矩陣

本 notebook 使用 Excel 的 `COVARIANCE.P` / `VAR.P` 對應口徑：

- 年化平均報酬 = 日平均報酬 × 241  
- 年化 covariance matrix = 日 covariance matrix × 241  
- `returns.cov(ddof=0)` 等同 population covariance，也就是 Excel 的 `COVARIANCE.P`


In [ ]:
mu_annual = returns.mean() * ANNUAL_FACTOR
cov_annual = returns.cov(ddof=0) * ANNUAL_FACTOR

print("年化平均報酬：")
display((mu_annual * 100).round(4).astype(str) + "%")

print("年化 covariance matrix：")
display(cov_annual.round(6))


## 四、建立完整 mean-variance frontier

對於給定目標報酬 `m`，最小變異數投資組合可由 Markowitz 解析式計算。  
定義：

- `A = 1′Σ⁻¹1`
- `B = 1′Σ⁻¹μ`
- `C = μ′Σ⁻¹μ`
- `D = AC − B²`

則目標報酬 `m` 對應的最小變異數為：

`σ²(m) = (A m² − 2B m + C) / D`

本作業圖形以不同目標報酬 `m` 掃描出完整 mean-variance frontier。


In [ ]:
def frontier_stats(mu, Sigma):
    mu = np.asarray(mu, dtype=float)
    Sigma = np.asarray(Sigma, dtype=float)

    inv = np.linalg.inv(Sigma)
    ones = np.ones(len(mu))

    A = ones @ inv @ ones
    B = ones @ inv @ mu
    C = mu @ inv @ mu
    D = A * C - B ** 2

    w_gmv = inv @ ones / A
    r_gmv = B / A
    s_gmv = math.sqrt(1 / A)

    return {
        "mu": mu,
        "Sigma": Sigma,
        "inv": inv,
        "ones": ones,
        "A": A,
        "B": B,
        "C": C,
        "D": D,
        "w_gmv": w_gmv,
        "r_gmv": r_gmv,
        "s_gmv": s_gmv,
    }

def target_variance(target_return, st):
    A, B, C, D = st["A"], st["B"], st["C"], st["D"]
    return (A * target_return ** 2 - 2 * B * target_return + C) / D

def target_sigma(target_return, st):
    return math.sqrt(max(target_variance(target_return, st), 0))

def upper_return_at_sigma(sigma, st):
    A, B, D = st["A"], st["B"], st["D"]
    discr = D * (A * sigma ** 2 - 1)
    if discr < -1e-12:
        return np.nan
    return (B + math.sqrt(max(discr, 0))) / A

def make_frontier_points(st, lower=0.15, upper=0.85, n=500):
    targets = np.linspace(lower, upper, n)
    sigmas = np.array([target_sigma(t, st) for t in targets])
    return pd.DataFrame({"Sigma(P)": sigmas, "E(P)": targets})

st3 = frontier_stats(mu_annual[COMMON_3], cov_annual.loc[COMMON_3, COMMON_3])
st4 = frontier_stats(mu_annual[ASSETS_4], cov_annual.loc[ASSETS_4, ASSETS_4])

frontier_3 = make_frontier_points(st3, lower=0.15, upper=0.75)
frontier_4 = make_frontier_points(st4, lower=0.15, upper=0.85)


## 五、輸出主要比較結果

以下結果是報告中使用的正確數據。


In [ ]:
def pct(x, digits=2):
    return f"{x * 100:.{digits}f}%"

gmv_summary = pd.DataFrame({
    "比較項目": ["GMV 年化標準差 Sigma(P)", "GMV 年化期望報酬 E(P)"],
    "三家公司": [pct(st3["s_gmv"]), pct(st3["r_gmv"])],
    "四家公司": [pct(st4["s_gmv"]), pct(st4["r_gmv"])],
    "改善幅度": [
        f"風險下降 {(st3['s_gmv'] - st4['s_gmv']) * 100:.2f} 個百分點",
        f"預期報酬提高 {(st4['r_gmv'] - st3['r_gmv']) * 100:.2f} 個百分點",
    ],
})

same_risk = pd.DataFrame([
    {
        "相同風險": pct(s, 0),
        "三家公司 E(P)": pct(upper_return_at_sigma(s, st3)),
        "四家公司 E(P)": pct(upper_return_at_sigma(s, st4)),
        "改善幅度": f"{(upper_return_at_sigma(s, st4) - upper_return_at_sigma(s, st3)) * 100:.2f} 個百分點",
    }
    for s in [0.22, 0.25]
])

same_return = pd.DataFrame([
    {
        "相同預期報酬": pct(r, 0),
        "三家公司 Sigma(P)": pct(target_sigma(r, st3)),
        "四家公司 Sigma(P)": pct(target_sigma(r, st4)),
        "改善幅度": f"{(target_sigma(r, st3) - target_sigma(r, st4)) * 100:.2f} 個百分點",
    }
    for r in [0.45, 0.50]
])

print("GMV 比較")
display(gmv_summary)

print("相同風險比較")
display(same_risk)

print("相同預期報酬比較")
display(same_return)

print("三家公司 GMV 權重")
display(pd.Series(st3["w_gmv"], index=COMMON_3).to_frame("weight"))

print("四家公司 GMV 權重")
display(pd.Series(st4["w_gmv"], index=ASSETS_4).to_frame("weight"))


## 六、繪製三張效率前緣圖

圖形使用相同座標口徑：

- 橫軸：年化標準差 `Sigma(P)`
- 縱軸：年化期望報酬 `E(P)`
- 保留 GMV 上下兩支曲線，因此是完整 mean-variance frontier


In [ ]:
def plot_frontier(st, frontier, assets, title, file_name):
    fig, ax = plt.subplots(figsize=(8.6, 5.6), dpi=160)

    ax.plot(frontier["Sigma(P)"], frontier["E(P)"], linewidth=2, label="完整 mean-variance frontier")
    ax.scatter(st["s_gmv"], st["r_gmv"], s=55, marker="o", label="GMV 最小變異組合")
    ax.annotate("GMV", (st["s_gmv"], st["r_gmv"]), xytext=(8, 8), textcoords="offset points")

    for asset in assets:
        sigma_i = math.sqrt(cov_annual.loc[asset, asset])
        return_i = mu_annual[asset]
        ax.scatter(sigma_i, return_i, marker="x", s=40)
        ax.annotate(asset.split()[1], (sigma_i, return_i), xytext=(6, 6), textcoords="offset points")

    ax.set_title(title)
    ax.set_xlabel("年化標準差 Sigma(P)")
    ax.set_ylabel("年化期望報酬 E(P)")
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))

    # 這裡設定的是主要視覺化區間，不代表 frontier 只有這段。
    ax.set_xlim(0.15, 0.60)
    ax.set_ylim(0.10, 0.90)

    ax.grid(True, alpha=0.28)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(file_name, bbox_inches="tight")
    plt.show()

def plot_overlay(file_name):
    fig, ax = plt.subplots(figsize=(8.6, 5.6), dpi=160)

    ax.plot(frontier_3["Sigma(P)"], frontier_3["E(P)"], linewidth=2, label="三家公司完整 frontier")
    ax.plot(frontier_4["Sigma(P)"], frontier_4["E(P)"], linewidth=2, label="四家公司完整 frontier")

    ax.scatter(st3["s_gmv"], st3["r_gmv"], s=55, marker="o", label="三家公司 GMV")
    ax.scatter(st4["s_gmv"], st4["r_gmv"], s=55, marker="s", label="四家公司 GMV")

    ax.annotate("三家 GMV", (st3["s_gmv"], st3["r_gmv"]), xytext=(8, 8), textcoords="offset points")
    ax.annotate("四家 GMV", (st4["s_gmv"], st4["r_gmv"]), xytext=(8, 8), textcoords="offset points")

    ax.set_title("三家公司與四家公司完整 mean-variance frontier 疊合圖")
    ax.set_xlabel("年化標準差 Sigma(P)")
    ax.set_ylabel("年化期望報酬 E(P)")
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlim(0.15, 0.60)
    ax.set_ylim(0.10, 0.90)

    ax.grid(True, alpha=0.28)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(file_name, bbox_inches="tight")
    plt.show()

plot_frontier(st3, frontier_3, COMMON_3, "三家公司完整 mean-variance frontier", "frontier_3_assets.png")
plot_frontier(st4, frontier_4, ASSETS_4, "四家公司完整 mean-variance frontier", "frontier_4_assets.png")
plot_overlay("frontier_overlay_3_vs_4_assets.png")


## 七、匯出結果表

這些 CSV 可用於檢查或貼入報告。


In [ ]:
prices.to_csv("merged_prices_2024.csv", index=False, encoding="utf-8-sig")
returns.to_csv("simple_returns_2024.csv", encoding="utf-8-sig")
mu_annual.to_frame("Annual_Mean_Return").to_csv("annual_mean_returns.csv", encoding="utf-8-sig")
cov_annual.to_csv("annual_covariance_matrix_population.csv", encoding="utf-8-sig")
gmv_summary.to_csv("gmv_summary.csv", index=False, encoding="utf-8-sig")
same_risk.to_csv("same_risk_comparison.csv", index=False, encoding="utf-8-sig")
same_return.to_csv("same_return_comparison.csv", index=False, encoding="utf-8-sig")

print("已輸出資料表與三張效率前緣圖。")
